# Nonlinear System Solvers (Newton-Raphson Method)

**References:** `Notes/Newton_SOE.ipynb` (Professor code), `4/Newton_Systems.ipynb`, `Notes/Quiz2/08_ODE_Systems_and_Nonlinear_Solvers.ipynb`

## Concept & Formulas
Find the root $x^*$ (or equilibrium point of an ODE) where $F(x) = 0$.

- **1D Update:** $x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}$.
- **Multi-Variable System ($F(x) = 0$):** Solve the linear system at each step:
  $$J(x_k)\, \Delta x = -F(x_k), \qquad x_{k+1} = x_k + \Delta x$$
  where $J_{ij} = \frac{\partial F_i}{\partial x_j}$ is the Jacobian matrix.
- **Convergence:** Quadratic (doubles correct digits each step) near a simple root.
- **Limitations:** Requires a good initial guess and fails if $J(x_k)$ is singular ($\det J = 0$).

In [1]:
import numpy as np

# Simple class version (from 4/Newton_Systems.ipynb style)
# Solve:  x^2 + y^2 - 1 = 0  and  y - x^3 = 0

def F(x):
    return np.array([x[1] - x[0]**3,
                     x[0]**2 + x[1]**2 - 1.0])

def J(x):
    return np.array([[-3.0 * x[0]**2, 1.0],
                     [2.0 * x[0],     2.0 * x[1]]])

x = np.array([1.0, 2.0])  # initial guess
tol = 1e-8

for step in range(20):
    dx = np.linalg.solve(J(x), -F(x))
    x = x + dx
    if np.linalg.norm(dx) < tol:
        break

print(f'Converged in {step + 1} iterations')
print('Solution x =', np.round(x, 6))
print('Check F(x) =', np.round(F(x), 10))

Converged in 6 iterations
Solution x = [0.826031 0.563624]
Check F(x) = [-0.  0.]


---
## Professor's Code (`Notes/Newton_SOE.ipynb`)
Exact implementation using manual Gaussian elimination with partial pivoting.

In [2]:
import numpy as np


def func(x):
    return -1 * np.array([
        x[1] - x[0]**3,
        x[0]**2 + x[1]**2 - 1
    ])


# Initial guess
x = np.array([1.0, 2.0])

# Jacobian matrix
a = np.array([[-3 * x[0]**2, 1],[2 * x[0], 2 * x[1]]], dtype=float)

# Function vector
f = func(x)

# Number of variables
n = a.shape[0]

# Tolerance
tol = n * 1e-2

# Difference between successive x vectors
dx = np.ones(n)


while np.sum(dx) > tol:

    xprev = x.copy()

    # Gaussian elimination with partial pivoting
    for j in range(n - 1):

        # Find maximum element for pivoting
        I = j + np.argmax(np.abs(a[j:n, j]))

        # Row swapping
        if I != j:
            a[[j, I], :] = a[[I, j], :]
            f[[j, I]] = f[[I, j]]

        # Elimination
        for i in range(j + 1, n):

            # Multiplication factor
            mult = a[i, j] / a[j, j]

            for k in range(j, n):
                a[i, k] = a[i, k] - mult * a[j, k]

            f[i] = f[i] - mult * f[j]

    # Back substitution
    s = np.zeros(n)

    # Last variable
    s[n - 1] = f[n - 1] / a[n - 1, n - 1]

    x[n - 1] = x[n - 1] + s[n - 1]

    # Remaining variables
    for i in range(n - 2, -1, -1):

        SUM = 0.0

        for j in range(i + 1, n):
            SUM = SUM + a[i, j] * s[j]

        s[i] = (f[i] - SUM) / a[i, i]

        x[i] = x[i] + s[i]

    # Update Jacobian
    a = np.array([
        [-3 * x[0]**2, 1],
        [2 * x[0], 2 * x[1]]
    ], dtype=float)

    # Update function vector
    f = func(x)

    # Calculate difference between iterations
    dx = np.abs(xprev - x)


# Display solution
print("Solution:")
print(x)

Solution:
[0.82604011 0.56361977]
